# Echo Level Guesser: Evaluation and Handoff

**Status: experimental baseline, not connected to production drafting.**

We estimate the difficulty of a candidate using five nearest bank problems in the
same topic. Three of five must agree; otherwise the estimator abstains. Existing
Level 1–4 labels are retained, not reassigned.

**Important plan gap:** this first adapter uses LaTeX source-structure proxies,
not semantic expression ASTs. It is not the completed level gate in `PLAN.md`.
The source bank, answer checker and approved plan remain unchanged.

In [1]:
from pathlib import Path
import sys, json, subprocess
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "docs/verified_problems.json").exists())
sys.path.insert(0, str(root))
local_python = root / ".venv" / ("Scripts/python.exe" if sys.platform == "win32" else "bin/python")
python = str(local_python) if local_python.exists() else sys.executable
run = subprocess.run([python, "scripts/evaluate_levels.py"], cwd=root, capture_output=True, text=True, check=True)
print(run.stdout)
report = json.loads((root / "build/level_guesser/evaluation.json").read_text())


{
  "bank_rows": 3510,
  "feature_supported": 3318,
  "feature_topics": 86,
  "excluded_rows": 192,
  "predicted": 1459,
  "abstained": 1859,
  "correct": 746,
  "coverage": 0.4397227245328511,
  "accuracy_all": 0.22483423749246534,
  "accuracy_when_predicted": 0.5113091158327622,
  "within_one_when_predicted": 0.8567511994516792,
  "majority_baseline_accuracy_all": 0.06630500301386377
}
Results: ./build/level_guesser



## 1. What is measured?

Five-fold holdout groups exact statement text and number-masked variants together.
Each query uses only same-topic references from the other folds. Feature scaling
is computed from those references, never from held-out examples. Mathematical
equivalents that the text grouping misses may still leak across folds.

Accuracy is agreement with one tutor's labels, not an independent proof of
pedagogical difficulty. No real AI-generated test set is used.

In [2]:
s = report["summary"]
print(f"Feature-supported bank problems: {s['feature_supported']} across {s['feature_topics']} topics")
print(f"Estimates returned: {s['predicted']} / {s['feature_supported']} ({s['coverage']:.1%})")
print(f"Abstentions: {s['abstained']}")
print(f"Exact accuracy when predicted: {s['accuracy_when_predicted']:.1%}")
print(f"Exact accuracy over all held-out problems: {s['accuracy_all']:.1%}")
print(f"Within one level, when predicted: {s['within_one_when_predicted']:.1%}")
print(f"Training-topic-majority baseline, all problems: {s['majority_baseline_accuracy_all']:.1%}")
print("Abstention reasons:", report["abstention_reasons"])


Feature-supported bank problems: 3318 across 86 topics
Estimates returned: 1459 / 3318 (44.0%)
Abstentions: 1859
Exact accuracy when predicted: 51.1%
Exact accuracy over all held-out problems: 22.5%
Within one level, when predicted: 85.7%
Training-topic-majority baseline, all problems: 6.6%
Abstention reasons: {'neighbor_boundary_tie': 1427, 'no_majority': 432}


## 2. Inspect errors, not just a headline accuracy

The matrix below includes returned estimates only. Abstentions remain in the
all-problem denominator above. Rows are the bank's true level; columns are the
predicted level. The grouped split can substantially change training-label
balance, so the simple majority baseline is specific to this evaluation.

In [3]:
matrix = report["confusion_matrix"]
print("True level -> predicted counts [L1, L2, L3, L4]")
for level, row in zip(matrix["row_true_levels"], matrix["counts"]):
    print(f"L{level}: {row}")
print("\nFirst five incorrect returned estimates:")
wrong = [r for r in report["predictions"] if r["predicted_level"] is not None and not r["correct"]]
for row in wrong[:5]:
    print(f"{row['topic']}::{row['id']} | bank L{row['true_level']} | predicted L{row['predicted_level']}")


True level -> predicted counts [L1, L2, L3, L4]
L1: [242, 76, 34, 16]
L2: [84, 158, 72, 37]
L3: [56, 76, 125, 98]
L4: [29, 37, 98, 221]

First five incorrect returned estimates:
at_infinity::LI06 | bank L3 | predicted L2
at_infinity::LI28 | bank L3 | predicted L4
classify_discontinuity::CD01 | bank L3 | predicted L4
classify_discontinuity::CD03 | bank L3 | predicted L4
classify_discontinuity::CD08 | bank L3 | predicted L2


## 3. Try a candidate without leaking the requested level into its estimate

This is a local demonstration using a saved candidate, not a new live AI call.
The answer key is supplied as input, not validated by the level estimator. The
result exposes neighbor levels and distances, including why it abstains.

In [4]:
from levels.guesser import LevelGuesser, prepare
from scripts.audit_bank import load_bank
references, excluded = prepare(load_bank(root / "docs/verified_problems.json"))
guesser = LevelGuesser(references)
statement = "Find $f'(x)$ for $f(x) = 4x^{3} - 5x^{2} + 3x - 7$."
key = "12x^{2} - 10x + 3"
result = guesser.check_requested_level("power_rule", statement, key, requested_level=3)
print(json.dumps(result, indent=2))


{
  "status": "estimated",
  "estimated_level": 1,
  "reason": "Strict majority of five nearest same-topic references",
  "votes": {
    "1": 3,
    "2": 2
  },
  "vote_fraction": 0.6,
  "features": {
    "problem_nodes": 20.0,
    "nesting_depth": 1.0,
    "distinct_functions": 0.0,
    "key_nodes": 9.0
  },
  "neighbors": [
    {
      "topic": "power_rule",
      "id": "PR31",
      "level": 2,
      "distance": 0.0
    },
    {
      "topic": "power_rule",
      "id": "PR03",
      "level": 2,
      "distance": 0.766034
    },
    {
      "topic": "power_rule",
      "id": "PR10",
      "level": 1,
      "distance": 0.766034
    },
    {
      "topic": "power_rule",
      "id": "PR01",
      "level": 1,
      "distance": 0.975515
    },
    {
      "topic": "power_rule",
      "id": "PR39",
      "level": 1,
      "distance": 0.975515
    }
  ],
  "warning": "Experimental source-structure baseline; votes are not calibrated confidence or proof of difficulty.",
  "requested_level": 3

## 4. Handoff recommendation

The measured coverage and accuracy are too limited to claim that the estimator
reliably enforces requested difficulty. Keep this separate from the drafting
pipeline pending Josh's review. The next substantive improvements are typed
expression features, task-aware adapters, independent ratings, and a separate
external evaluation—not a promise that nearest neighbors always get the level right.

In [5]:
for limitation in report["limitations"]:
    print("-", limitation)
print("\nSource bank SHA-256:", report["bank_sha256"])
print("Implementation: levels/guesser.py")
print("Evaluation: scripts/evaluate_levels.py")
print("Handoff: levels/README.md")


- Source tokens and delimiter nesting are proxies, not semantic expression ASTs required by the final plan.
- Eligibility reuses heuristic audit only; per-topic task/answer-type adapters are not implemented.
- Same-family variants are held together, but undetected mathematical equivalents may still cross folds.
- Targets are one tutor’s bank ratings, not independent ground truth.
- No accuracy claim is made for new AI-generated problems.
- This module is not wired into core.entry or drafting; bank eligibility/checker behavior is unchanged.

Source bank SHA-256: b254e1cb2e55be28740a37f88cb167de72391ff52fb06861107f4f04e72f2007
Implementation: levels/guesser.py
Evaluation: scripts/evaluate_levels.py
Handoff: levels/README.md
